In [ ]:
from pathlib import Path
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
)

current_path = Path.cwd()

project_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").exists()
)

data_path = (
    project_root
    / "data"
    / "preprocessing"
    / "train_processed.csv"
)

dataset = pd.read_csv(data_path, index_col=0)

X = dataset.drop(columns=["Attrition"])
y = dataset["Attrition"].map({0: 1, 1: 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def objective(trial):
 params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000, step=50
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10.0, log=True
        ),

        # 고정 설정
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,  # GPU 한 장에 여러 모델을 동시에 올리지 않는다.
    )

    return scores.mean()
    
    # 4. 평균 성능 반환 (Optuna는 이 값을 최대화/최소화 시킴)
    return score.mean()

d:\SK_encoa\SKN35-2nd-5Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
